# 217. Contains Duplicate
**Difficulty:** 🟢 Easy · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/contains-duplicate/

## 💡 Concepts

**Core concept(s):** Membership tracking with a **hash set**; alternatively, **sorting** to make duplicates adjacent.

**Why they apply here:** A duplicate exists iff, while scanning, we ever meet a value we've already recorded. A hash set answers "seen before?" in O(1). Sorting is an alternative: identical values become neighbors, so one linear pass over the sorted array finds them.

**Key intuition / mental model:** Trade memory for speed — remember every value you pass in a set and stop the instant one repeats.

---

### 📚 What is a Hash Set?
A **hash set** (Python `set`) stores unique elements and answers "is `x` present?" by hashing `x` to a bucket — no scanning.
- **Operations & complexity:** add / lookup / delete are **O(1) average** (O(n) worst case under many collisions).
- **In Python:** `set()`; add with `.add(x)`, test with `x in s`.

### 📚 What is Sorting (as a tool)?
**Sorting** reorders elements so structure (duplicates, closeness, order statistics) becomes easy to exploit in one pass.
- **Complexity:** Python's `sorted()` / `.sort()` (Timsort) is **O(n log n)** time; `sorted()` uses O(n) extra space, `list.sort()` sorts in place.

## 📝 Problem

Return `True` if any value appears **at least twice** in `nums`, else `False`.

**Example**
```
Input:  nums = [1, 2, 3, 1]      Output: True
Input:  nums = [1, 2, 3, 4]      Output: False
```
**Constraints:** `1 <= len(nums) <= 10^5`.

### Approach 1 — Brute Force (worst)

**Idea:** Compare every pair `(i, j)`; return `True` on the first match.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)`.

In [ ]:
from typing import List

def contains_duplicate_brute(nums: List[int]) -> bool:
    n = len(nums)
    for i in range(n):                     # compare every pair of positions...
        for j in range(i + 1, n):
            if nums[i] == nums[j]:         # found two equal values
                return True
    return False

### Approach 2 — Sort + Scan Neighbors (better)

**Idea:** Sort, then any duplicate must sit next to its twin — check adjacent pairs.

**Time complexity:** `O(n log n)` — dominated by the sort.

**Space complexity:** `O(n)` for `sorted()` (or O(1) with in-place `sort()`).

In [ ]:
from typing import List

def contains_duplicate_sort(nums: List[int]) -> bool:
    s = sorted(nums)                       # after sorting, equal values sit next to each other
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:               # a value equals its neighbor -> duplicate
            return True
    return False

### Approach 3 — Hash Set (optimal)

**Idea:** Scan once; if a value is already in the set, it's a duplicate; otherwise add it.

**Time complexity:** `O(n)` average.

**Space complexity:** `O(n)`.

In [ ]:
from typing import List

def contains_duplicate_set(nums: List[int]) -> bool:
    seen = set()                           # values we've already encountered
    for x in nums:
        if x in seen:                      # already met this value -> duplicate
            return True
        seen.add(x)                        # otherwise remember it and continue
    return False

In [ ]:
# Correctness check
tests = [
    ([1, 2, 3, 1], True),
    ([1, 2, 3, 4], False),
    ([1, 1, 1, 3, 3, 4, 3, 2, 4, 2], True),
    ([1], False),
]
for nums, expected in tests:
    b, s, h = contains_duplicate_brute(nums), contains_duplicate_sort(nums), contains_duplicate_set(nums)
    print(f"{nums} -> brute={b}, sort={s}, set={h} | expected={expected}")
    assert b == s == h == expected, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit) so the measurement reflects the true bound. Sub-millisecond rows are noisy — look at the trend, not one number.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    nums = list(range(n))                  # all distinct -> no early exit for any approach
    return (nums,)

solutions = {
    "brute O(n^2)   ": contains_duplicate_brute,
    "sort  O(n log n)": contains_duplicate_sort,
    "set   O(n)     ": contains_duplicate_set,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Hash set for "seen before?":** Any time you need to detect repeats or test membership inside a loop, a set turns an O(n²) pairwise scan into O(n).
- **Sort-to-group:** Sorting makes equal/near values adjacent — a fallback when O(1) extra space matters more than the log factor.
- **Signal to reach for it:** "contains duplicate", "all unique?", "first repeated element", "intersection of arrays".
- **Related problems:** Two Sum, Longest Consecutive Sequence, Group Anagrams, Contains Duplicate II/III.
- **Common pitfalls:** (1) using a list instead of a set for membership (`x in list` is O(n)!); (2) forgetting sorting costs O(n log n); (3) mutating the input with in-place sort when the caller needs original order.